# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/notebooks/03_working_with_the_full_release.ipynb?flush_cache=true)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


In [1]:
%pip -q install duckdb huggingface_hub


In [2]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


Paste your Hugging Face READ token (hf_...): ··········


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [3]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [4]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


clients with 12+ months of GSC history: 4


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [5]:
# 90-day feature engineering
features_90d = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d
        FROM {TABLES['fact_daily']}
    ),

    windowed AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,

            -- Outcome window: most recent 30 days
            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_impressions
                    ELSE 0
                END
            ) AS imp_last30,

            -- Previous 30 days
            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 60 DAY
                     AND f.report_date <= b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_impressions
                    ELSE 0
                END
            ) AS imp_prev30,

            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 60 DAY
                     AND f.report_date <= b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_clicks
                    ELSE 0
                END
            ) AS clk_prev30,

            -- Older 30-day history
            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 90 DAY
                     AND f.report_date <= b.end_d - INTERVAL 60 DAY
                    THEN f.gsc_impressions
                    ELSE 0
                END
            ) AS imp_prev_prev30,

            -- Last-30 clicks (used only for inspection, not prediction)
            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_clicks
                    ELSE 0
                END
            ) AS clk_last30,

            -- Previous-period average position
            AVG(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 60 DAY
                     AND f.report_date <= b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_avg_position
                END
            ) AS pos_prev30,

            -- Position volatility from PREVIOUS 30 days
            STDDEV_SAMP(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 60 DAY
                     AND f.report_date <= b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_avg_position
                END
            ) AS pos_volatility

        FROM {TABLES['fact_daily']} f, bounds b

        WHERE f.report_date > b.end_d - INTERVAL 90 DAY

        GROUP BY 1, 2

        HAVING imp_prev30 >= 200
    )

    SELECT
        *,

        -- Historical impression trend
        CASE
            WHEN imp_prev_prev30 > 0
            THEN imp_prev30 / imp_prev_prev30
            ELSE NULL
        END AS impression_trend,

        -- Previous 30-day CTR
        CASE
            WHEN imp_prev30 > 0
            THEN clk_prev30 / imp_prev30
            ELSE NULL
        END AS ctr_prev30

    FROM windowed
""").df()

print(f"{len(features_90d):,} content items with enough history")

print("\nColumns:")
print(features_90d.columns.tolist())

print("\nFirst 5 rows:")
display(features_90d.head())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

88,301 content items with enough history

Columns:
['client_hash_id', 'content_hash_id', 'imp_last30', 'imp_prev30', 'clk_prev30', 'imp_prev_prev30', 'clk_last30', 'pos_prev30', 'pos_volatility', 'impression_trend', 'ctr_prev30']

First 5 rows:


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_prev30,imp_prev_prev30,clk_last30,pos_prev30,pos_volatility,impression_trend,ctr_prev30
0,client_e547b89c05043229,content_25dfa3e39bc37247,216.0,616.0,5.0,444.0,1.0,8.502874,3.243938,1.387387,0.008117
1,client_e547b89c05043229,content_a64be0ba11772089,248.0,409.0,2.0,448.0,0.0,17.281978,8.333510,0.912946,0.004890
2,client_e547b89c05043229,content_74369d7d3369d86b,192.0,358.0,1.0,778.0,2.0,19.978176,8.733430,0.460154,0.002793
3,client_e547b89c05043229,content_1e1be93551c6cb91,629.0,676.0,1.0,722.0,2.0,20.496973,8.043831,0.936288,0.001479
4,client_e547b89c05043229,content_a48e7445655ac0f2,653.0,511.0,0.0,474.0,0.0,79.580511,6.312109,1.078059,0.000000


## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [6]:
# Query signals + merge

qsignals = con.sql(f"""
    SELECT
        content_hash_id,

        ANY_VALUE(content_visible_query_count) AS visible_queries,

        ANY_VALUE(rare_impressions_share) AS rare_share,

        ANY_VALUE(anonymized_impressions_share) AS anon_share,

        MAX(impressions_90d) AS top_query_impressions,

        SUM(impressions_90d) AS kept_impressions

    FROM {TABLES['fact_query_90d']}

    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = (
    qsignals['top_query_impressions']
    / qsignals['kept_impressions']
)

data = features_90d.merge(
    qsignals,
    on='content_hash_id',
    how='left'
)

print(f"Feature rows: {len(features_90d):,}")
print(f"Query-signal rows: {len(qsignals):,}")
print(f"Joined rows: {len(data):,}")

print("\nFinal columns:")
print(data.columns.tolist())

display(data.head())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature rows: 88,301
Query-signal rows: 133,852
Joined rows: 88,301

Final columns:
['client_hash_id', 'content_hash_id', 'imp_last30', 'imp_prev30', 'clk_prev30', 'imp_prev_prev30', 'clk_last30', 'pos_prev30', 'pos_volatility', 'impression_trend', 'ctr_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_impressions', 'kept_impressions', 'top_query_share']


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_prev30,imp_prev_prev30,clk_last30,pos_prev30,pos_volatility,impression_trend,ctr_prev30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_e547b89c05043229,content_25dfa3e39bc37247,216.0,616.0,5.0,444.0,1.0,8.502874,3.243938,1.387387,0.008117,5.0,0.038401,0.897335,34.0,82.0,0.414634
1,client_e547b89c05043229,content_a64be0ba11772089,248.0,409.0,2.0,448.0,0.0,17.281978,8.333510,0.912946,0.004890,1.0,0.081448,0.902262,18.0,18.0,1.000000
2,client_e547b89c05043229,content_74369d7d3369d86b,192.0,358.0,1.0,778.0,2.0,19.978176,8.733430,0.460154,0.002793,9.0,0.054970,0.509036,401.0,579.0,0.692573
3,client_e547b89c05043229,content_1e1be93551c6cb91,629.0,676.0,1.0,722.0,2.0,20.496973,8.043831,0.936288,0.001479,11.0,0.076961,0.745930,88.0,359.0,0.245125
4,client_e547b89c05043229,content_a48e7445655ac0f2,653.0,511.0,0.0,474.0,0.0,79.580511,6.312109,1.078059,0.000000,23.0,0.085470,0.141636,117.0,1266.0,0.092417


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


In [7]:
# Improved GroupShuffleSplit model
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, accuracy_score, balanced_accuracy_score, roc_auc_score, confusion_matrix

# Target
data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)


# Features
feature_cols = [
    'imp_prev30',
    'visible_queries',
    'rare_share',
    'anon_share',
    'top_query_share',
    'pos_volatility',
    'impression_trend',
    'ctr_prev30'
]

print("Features:")
for col in feature_cols:
    print("-", col)


# Remove missing values
model_data = data.dropna(subset=feature_cols + ['client_hash_id']).copy()

X = model_data[feature_cols]
y = model_data['is_declining']
groups = model_data['client_hash_id']


# Group split by client
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)

train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_tr = X.iloc[train_idx]
X_te = X.iloc[test_idx]

y_tr = y.iloc[train_idx]
y_te = y.iloc[test_idx]

groups_tr = groups.iloc[train_idx]
groups_te = groups.iloc[test_idx]


# Verify client separation
train_clients = set(groups_tr)
test_clients = set(groups_te)

print("\nSplit:")
print(f"Training rows:  {len(X_tr):,}")
print(f"Testing rows:   {len(X_te):,}")
print(f"Training clients: {len(train_clients):,}")
print(f"Testing clients:  {len(test_clients):,}")

print("Overlapping clients:", len(train_clients.intersection(test_clients)))


# Train model
print("\nTraining Random Forest...")

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)

model.fit(X_tr, y_tr)


# Predictions
y_pred = model.predict(X_te)
y_prob = model.predict_proba(X_te)[:, 1]


# Metrics
majority_class = y_te.mode()[0]

baseline_accuracy = (y_te == majority_class).mean()

accuracy = accuracy_score(y_te, y_pred)

balanced_acc = balanced_accuracy_score(y_te, y_pred)

roc_auc = roc_auc_score(y_te, y_prob)


print("\n============================================")
print("IMPROVED GROUPSHUFFLESPLIT RESULTS")
print("============================================")

print(f"Baseline Accuracy:   {baseline_accuracy:.3f}")
print(f"Model Accuracy:      {accuracy:.3f}")
print(f"Balanced Accuracy:   {balanced_acc:.3f}")
print(f"ROC-AUC:             {roc_auc:.3f}")


# Classification report
print("\nClassification Report:")
print(classification_report(y_te, y_pred, digits=3))


# Confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_te, y_pred))


# Feature importance
feature_importance = sorted(
    zip(
        feature_cols,
        model.feature_importances_
    ),
    key=lambda x: x[1],
    reverse=True
)

print("\nFeature Importance:")
print("============================================")

for feature, importance in feature_importance:
    print(
        f"{feature:25s} {importance:.4f}"
    )

Features:
- imp_prev30
- visible_queries
- rare_share
- anon_share
- top_query_share
- pos_volatility
- impression_trend
- ctr_prev30

Split:
Training rows:  61,007
Testing rows:   17,251
Training clients: 33
Testing clients:  12
Overlapping clients: 0

Training Random Forest...

IMPROVED GROUPSHUFFLESPLIT RESULTS
Baseline Accuracy:   0.584
Model Accuracy:      0.590
Balanced Accuracy:   0.528
ROC-AUC:             0.612

Classification Report:
              precision    recall  f1-score   support

           0      0.526     0.156     0.240      7182
           1      0.599     0.900     0.719     10069

    accuracy                          0.590     17251
   macro avg      0.563     0.528     0.480     17251
weighted avg      0.569     0.590     0.520     17251

Confusion Matrix:
[[1118 6064]
 [1006 9063]]

Feature Importance:
rare_share                0.1408
impression_trend          0.1359
pos_volatility            0.1319
imp_prev30                0.1295
anon_share                0

In [8]:
# ============================================
# FINAL FEATURE ANALYSIS
# ============================================

analysis_cols = [
    'imp_prev30',
    'visible_queries',
    'rare_share',
    'anon_share',
    'top_query_share',
    'pos_volatility',
    'impression_trend',
    'ctr_prev30'
]

comparison = data.groupby('is_declining')[analysis_cols].mean().T

comparison.columns = [
    'Not Declining (0)',
    'Declining (1)'
]

comparison['Difference'] = (
    comparison['Declining (1)']
    - comparison['Not Declining (0)']
)

display(comparison)

,Not Declining (0),Declining (1),Difference
imp_prev30,3027.878611,2748.823292,-279.055319
visible_queries,28.920894,25.463821,-3.457073
rare_share,0.106781,0.103490,-0.003291
anon_share,0.674326,0.687825,0.013498
top_query_share,0.333472,0.361130,0.027657
pos_volatility,6.490411,7.207207,0.716796
impression_trend,3.025698,2.151535,-0.874162
ctr_prev30,0.004551,0.002890,-0.001662


**My Observation**: A Random Forest model was trained to predict >20% month-over-month impression declines using features engineered from the full warehouse. A client-level GroupShuffleSplit was used to test generalization to unseen clients. The initial six-feature model achieved a ROC-AUC of 0.593, while adding historical impression trend and previous-period CTR improved ROC-AUC to 0.612. However, classification performance remained modest, with balanced accuracy of 0.528, indicating that the engineered signals provide some predictive value but do not yet generalize strongly across clients. Historical impression trend, query characteristics, and ranking volatility were among the most informative features.

Whatever number you just got: interrogate it before you believe it. Which feature carries the
signal? Does it survive a per-client split (train on some clients, test on others)? That
question — *does it generalize across clients?* — is exactly what separates a capstone-grade
result from a lucky split.

## Your turn

1. Re-run section 3 with a **90-day** window and a `HAVING` threshold of your choice.
2. Add one feature you believe in (position volatility? weekend share? query concentration?).
3. Replace the random split with **GroupShuffleSplit on `client_hash_id`** and compare.

## Working locally instead

```python
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])
```
Then point `REL` at that local path. Download only the month partitions you need — the
`allow_patterns` filter above is the whole trick.

---

**Where this fits:** every lane brief assumes you can produce per-content feature tables like
the one you just built. The lane datasets under the `lanes` HF repo are pre-cut examples of
exactly this pattern — but for the capstone, features you engineered yourself from the full
release beat any pre-cut file.
